In [ ]:
import langchain
import langgraph
import openai
import importlib
from dotenv import load_dotenv
load_dotenv()
print("LangChain版本：", langchain.__version__)
print("LangGraph版本：", importlib.metadata.version("langgraph"))
print("OpenAI版本：", openai.__version__)

In [ ]:
#在代码中读取密钥
import os
from langchain_openai import ChatOpenAI

load_dotenv() #读取.env文件中的变量
API_KEY=os.getenv("API_KEY")
BASE_URL=os.getenv("BASE_URL")

if not API_KEY:
    raise ValueError("未检测到 API_KEY，请检查 .env 文件是否配置正确")
if not BASE_URL:
    raise ValueError("未检测到 BASE_URL，请检查 .env 文件是否配置正确")

llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="mimo-v2.5-pro",  # 注意填写服务商指定的模型名称（见下方说明）
)
response = llm.invoke("你好，请回复'配置成功'")
print(response.content)

In [ ]:
#LangChain案例
#调用OpenAI模型,生成一段"AI学习建议"API_KEY

#构造prompt
prompt="请写一段50字左右的 AI 学习建议,语言简洁、实用，适合初学者。"

#调用模型
response=llm.invoke(prompt)

#输出结果
print("生成的学习建议: ")
print(response.content)

In [ ]:
#LangGraph案例:基础工作流执行

#1.导入需要的模块
import os
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph,START,END
from typing import TypedDict #约束字典的值必须为int
from dotenv import load_dotenv

#2.加载API密钥
load_dotenv()

#3.初始化大模型
API_KEY=os.getenv("API_KEY")
BASE_URL=os.getenv("BASE_URL")

if not API_KEY:
    raise ValueError("未检测到 API_KEY, 请检查 .env 文件是否配置正确")

#4.初始化大模型
#temperature 越低：回答越稳定、保守、确定
#temperature 越高：回答越发散、有创意、不稳定
llm=ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="mimo-v2.5-pro",
    temperature=0.3 
)

#5.定义State
class WorkflowState(TypedDict,total=False):
    user_role:str #存储用户角色
    original_advice:str #存储原始学习建议
    simplified_advice:str #存储精简后的建议

#6.定义节点
def generate_advice(state:WorkflowState):
    prompt=f"给{state['user_role']}写一段50字左右的 AI 学习建议。"
    result=llm.invoke(prompt)
    return {"original_advice": result.content}

def simplify_advice(state:WorkflowState):
    prompt=f"把下面的学习建议精简到30字以内: {state['original_advice']}"
    result=llm.invoke(prompt)
    return {"simplified_advice": result.content}

#7.构建工作流
workflow=StateGraph(WorkflowState)

#每个节点对应一个函数
workflow.add_node("generate",generate_advice)
workflow.add_node("simplify",simplify_advice)

#Edge:定义节点执行顺序
workflow.add_edge(START,"generate")
workflow.add_edge("generate","simplify")
workflow.add_edge("simplify",END)

#编译把状态图、节点、边的定义，转换成可执行的工作流实例；
app=workflow.compile()

#8.执行
result=app.invoke({"user_role":"高校学生"})

#9.输出
print("原始学习建议: ")
print(result["original_advice"])
print("\n精简后学习建议：")
print(result["simplified_advice"])

